In [1]:
import os
import time
import ffmpeg
import random
import subprocess

from appium import webdriver as appium_webdriver
from appium.webdriver.common.appiumby import AppiumBy
from appium.options.android import UiAutomator2Options

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from seleniumwire import webdriver as wire_webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from appium.webdriver.extensions.android.nativekey import AndroidKey
from selenium.webdriver.common.actions.pointer_input import PointerInput
from selenium.webdriver.common.actions.action_builder import ActionBuilder
from selenium.webdriver.common.desired_capabilities import DesiredCapabilities
from urllib.parse import urlparse, parse_qs, urlencode, urlunparse, quote_plus, unquote
from dotenv import load_dotenv

# Load .env file relative to this config.py
current_path = os.getcwd()
ENV_PATH = os.path.join(current_path, ".env")
print(ENV_PATH)
load_dotenv(dotenv_path=ENV_PATH, override=True)

# --- Environment Configuration ---
ENV = os.getenv("ENV", "production")
PORT = int(os.getenv("PORT", 5001))
WS_SCRCPY_PORT = int(os.getenv("WS_SCRCPY_PORT", 5001))
ADMIN_KEY = os.getenv("BACKEND_ADMIN_KEY")
TOKEN_EXPIRE_LIMIT = int(os.getenv("ADMIN_TOKEN_EXPIRE_LIMIT", 1))
DEV_API_KEY = os.getenv("BACKEND_DEV_API_KEY")
DEVICES_RANGE = os.getenv("DEVICES_RANGE", "192.168.1.0/24")
SECRET_KEY = os.getenv("FLASK_SECRET_KEY", "a_default_secret_key_if_not_set")

# --- Database (Postgres) ---
SQLALCHEMY_DATABASE_URI = (
    f"postgresql://{os.getenv('POSTGRES_USER')}:{os.getenv('POSTGRES_PASSWORD')}"
    f"@{os.getenv('POSTGRES_HOST')}:{os.getenv('POSTGRES_PORT')}/{os.getenv('POSTGRES_DB')}"
)
SQLALCHEMY_TRACK_MODIFICATIONS = False

# --- CORS Configuration ---
CORS_ORIGINS = os.getenv("CORS_ORIGINS", "").split(",")

# --- Clerk Authentication ---
CLERK_SECRET_KEY = os.getenv("CLERK_SECRET_KEY")
CLERK_ISSUER = os.getenv("CLERK_ISSUER")
CLERK_JWKS_URL = os.getenv("CLERK_JWKS_URL")

# --- Paths ---
BIN_FOLDER = os.path.join(current_path, "bin")
JSON_FOLDER = os.path.join(BIN_FOLDER, "files")
DATABASE_DIR = os.path.join(BIN_FOLDER, "database")
WS_SCRCPY_PATH = os.path.join(current_path, "ws-scrcpy")
ADB_SCRIPTS_PATH = os.path.join(current_path, "scripts")
FILE_UPLOAD_FOLDER = os.path.join(current_path, "uploads")

# Ensure directories exist
for folder in [BIN_FOLDER, JSON_FOLDER, DATABASE_DIR, FILE_UPLOAD_FOLDER]:
    if not os.path.exists(folder):
        os.makedirs(folder)

# --- JSON Databases ---
FILES_JSON_PATH = os.path.join(JSON_FOLDER, "files.json")
if os.path.exists(FILES_JSON_PATH):
    with open(FILES_JSON_PATH, "r") as f:
        UPLOADED_FILES_DATABASE = json.load(f)
else:
    UPLOADED_FILES_DATABASE = {}

APPIUM_CONTAINERS_DATABASE = os.path.join(JSON_FOLDER, "running_containers.json")

# --- Server Configuration ---
FLASK_RUN_PORT = int(os.getenv("BACKEND_PORT", 5000))

# --- Cloudflare Tunnels ---
CLOUDFLARED_TUNNEL_TOKEN = os.getenv("CLOUDFLARED_TUNNEL_TOKEN")

# --- Apps Package & Activities ---
YT_MUSIC_PACKAGE_NAME = os.getenv('YT_MUSIC_PACKAGE_NAME')
YT_MUSIC_ACTIVITY_NAME = os.getenv('YT_MUSIC_ACTIVITY_NAME')


# Skipping Rates
SKIPPING_RATES_COUNT = 8
SKIPPING_HIGH_COUNT  = 1
SKIPPING_LOW_RANGE   = [42,  58]
SKIPPING_HIGH_RANGE  = [100, 120]
SKIPPING_AVG_RANGE   = [55,  60]


device_id = "192.168.1.55:5555"

PACKAGE_ID = "com.google.android.apps.youtube.music" # "com.google.android.gms"
ACTIVITY = "com.google.android.apps.youtube.music.activities.MusicActivity" # 'com.google.android.gms/com.google.android.gms.auth.uiflows.minutemaid.MinuteMaidActivity' #
APPIUM_SERVER = "http://127.0.0.1:4723" 

/home/magician/Desktop/MagSpot/.env


In [2]:
def is_app_running(device_id, package_name):
    result = subprocess.getoutput(f"adb -s {device_id} shell pidof {package_name}")
    return bool(result.strip())

def create_driver(device_id, PACKAGE_ID, ACTIVITY):
    options = UiAutomator2Options().load_capabilities({
        "platformName": "Android",
        "automationName": "UiAutomator2",
        "udid": device_id,
        "deviceName": device_id,

        "appPackage": PACKAGE_ID,
        "appActivity": ACTIVITY,

        "autoLaunch": True,
        "forceAppLaunch": True,
        "noReset": True,

        "newCommandTimeout": 3600,
    })
    driver = appium_webdriver.Remote(command_executor=APPIUM_SERVER, options=options)
    if not is_app_running(device_id, PACKAGE_ID): driver.start_activity(PACKAGE_ID, ACTIVITY)
    time.sleep(4)
    return driver


def scroll_down(driver):    
    # get device screen size
    size = driver.get_window_size()
    width = size['width']
    height = size['height']
    # convert percentages to pixels
    start_x = int(width * 0.50)   # 50%
    start_y = int(height * 0.70)  # 70%
    end_x   = int(width * 0.50)   # 50%
    end_y   = int(height * 0.30)  # 30%
    # build gesture
    actions = ActionBuilder(driver, mouse=PointerInput("touch", "finger"))
    actions.pointer_action.move_to_location(start_x, start_y)  # move to start
    actions.pointer_action.pointer_down()                      # finger down
    actions.pointer_action.move_to_location(end_x, end_y)  # swipe
    actions.pointer_action.pointer_up()                        # finger up
    # perform action
    actions.perform()

In [3]:
import time
import random
import subprocess
import os, ffmpeg
from appium import webdriver as appium_webdriver
from appium.webdriver.common.appiumby import AppiumBy
from appium.options.android import UiAutomator2Options

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from seleniumwire import webdriver as wire_webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from appium.webdriver.extensions.android.nativekey import AndroidKey
from selenium.webdriver.common.actions.pointer_input import PointerInput
from selenium.webdriver.common.actions.action_builder import ActionBuilder
from selenium.webdriver.common.desired_capabilities import DesiredCapabilities
from urllib.parse import urlparse, parse_qs, urlencode, urlunparse, quote_plus, unquote


def tapOnScreenCoord(driver: appium_webdriver, x: int, y: int):
    """
    Tap on the screen at absolute coordinates (x, y).
    """
    driver.tap([(x, y)])
    time.sleep(1)


def tapOnScreen(driver: appium_webdriver, x_per=0.93, y_per=0.29):
    # get screen size
    size = driver.get_window_size()
    print(size)
    width = size['width']
    height = size['height']
    # convert percentages to pixels
    x = int(width * x_per)
    y = int(height * y_per)
    driver.tap([(x, y)])
    time.sleep(1)


def setBackgrounMusicVolume(driver, x_per=0.17, y_per=0.89):
    # Get screen size
    size = driver.get_window_size()
    width = size['width']
    height = size['height']
    # Convert percentages → pixels
    x = int(width * x_per)   # 17%
    y = int(height * y_per)  # 89%
    # Build tap gesture
    actions = ActionBuilder(driver, mouse=PointerInput("touch", "finger"))
    actions.pointer_action.move_to_location(x, y)
    actions.pointer_action.pointer_down()
    actions.pointer_action.pointer_up()
    actions.perform()
    time.sleep(1)


def scroll_up(driver):
    # get device screen size
    size = driver.get_window_size()
    width = size['width']
    height = size['height']
    # convert percentages to pixels
    start_x = int(width * 0.50)   # 50%
    start_y = int(height * 0.30)  # 70%
    end_x   = int(width * 0.50)   # 50%
    end_y   = int(height * 0.70)  # 30%
    # build gesture
    actions = ActionBuilder(driver, mouse=PointerInput("touch", "finger"))
    actions.pointer_action.move_to_location(start_x, start_y)  # move to start
    actions.pointer_action.pointer_down()                      # finger down
    actions.pointer_action.move_to_location(end_x, end_y, duration=1000)  # swipe
    actions.pointer_action.pointer_up()                        # finger up
    # perform action
    actions.perform()
    

def scroll_down(driver):    
    # get device screen size
    size = driver.get_window_size()
    width = size['width']
    height = size['height']
    # convert percentages to pixels
    start_x = int(width * 0.50)   # 50%
    start_y = int(height * 0.70)  # 70%
    end_x   = int(width * 0.50)   # 50%
    end_y   = int(height * 0.30)  # 30%
    # build gesture
    actions = ActionBuilder(driver, mouse=PointerInput("touch", "finger"))
    actions.pointer_action.move_to_location(start_x, start_y)  # move to start
    actions.pointer_action.pointer_down()                      # finger down
    actions.pointer_action.move_to_location(end_x, end_y, duration=1000)  # swipe
    actions.pointer_action.pointer_up()                        # finger up
    # perform action
    actions.perform()


def lower_video_quality(input_file, output_dir="compressed_videos", qualities=None):
    """
    Create lower-quality versions of a video.
    Args:
        input_file (str): Path to the input video.
        output_dir (str): Directory to save converted videos.
        qualities (dict): Mapping of resolution name -> scale height (width auto).
        Example: {"720p": 720, "480p": 480, "360p": 360}
    """
    if qualities is None: qualities = {"720p": 720, "480p": 480, "360p": 360}
    if not os.path.exists(output_dir): os.makedirs(output_dir)

    base_name = os.path.splitext(os.path.basename(input_file))[0]
    output_files = {}

    for label, height in qualities.items():
        out_path = os.path.join(output_dir, f"{base_name}_{label}.mp4")
        try:
            (
                ffmpeg
                .input(input_file)
                .output(out_path, vf=f"scale=-2:{height}", vcodec="libx264", crf=28, preset="fast")
                .overwrite_output()
                .run(quiet=True)
            )
            output_files[label] = out_path
            # print(f"✅ Saved {label} video: {out_path}")
        except Exception as e: print(f"❌ Failed to create {label} version: {e}")
    return output_files



def push_video_to_android(video_path, device_id):
    try:
        # STEP 1: Push Video To Android Phone
        local_file = video_path
        # pushed_file = os.path.join(FILE_UPLOAD_FOLDER, f"Video_{random.randint(1111,9999)}.mp4")
        # output_ = lower_video_quality(local_file, FILE_UPLOAD_FOLDER, qualities = {"720p": 720})
        output_ = {'720p': local_file}
        remote_path = f"/storage/emulated/0/Download/{os.path.basename(output_['720p'])}"

        # Push file
        subprocess.run(["adb", "-s", device_id, "push", output_['720p'], remote_path])

        # Trigger MediaScanner
        subprocess.run([
            "adb", "-s", device_id, "shell", 
            "am", "broadcast", "-a", "android.intent.action.MEDIA_SCANNER_SCAN_FILE", 
            "-d", f"file://{remote_path}"
        ])
        return remote_path, output_
    except Exception as e:
        raise Exception(f"ERROR: Failed To Push Video To Android {device_id} PATH:{video_path}. \n{e} --Output: {output_}")



def remove_video_from_android(android_video, device_id):
    subprocess.run([
        "adb", "-s", device_id, "shell", 
        "rm", f"/storage/emulated/0/Download/{os.path.basename(android_video['720p'])}"
    ])



def get_screen_resolution(driver):
    """
    Get the current screen resolution from an Appium driver.
    Returns (width, height) as integers.
    """
    size = driver.get_window_size()
    width = size['width']
    height = size['height']
    return width, height




In [4]:

# MAIN BUTTONS
HOME_BUTTON_XPATH1 = '//android.widget.TextView[@resource-id="com.google.android.apps.youtube.music:id/text1" and @text="Home"]'
HOME_BUTTON_XPATH2 = "//*[@resource-id='com.google.android.apps.youtube.music:id/pivot_bar']//*[@text='Home']"

SAMPLES_BUTTON_XPATH1 = '//android.widget.TextView[@resource-id="com.google.android.apps.youtube.music:id/text1" and @text="Samples"]'
SAMPLES_BUTTON_XPATH2 = "//*[@resource-id='com.google.android.apps.youtube.music:id/pivot_bar']//*[@text='Samples']"

EXPLORE_BUTTON_XPATH1 = '//android.widget.TextView[@resource-id="com.google.android.apps.youtube.music:id/text1" and @text="Explore"]'
EXPLORE_BUTTON_XPATH2 = "//*[@resource-id='com.google.android.apps.youtube.music:id/pivot_bar']//*[@text='Explore']"

LIBRARY_BUTTON_XPATH1 = '//android.widget.TextView[@resource-id="com.google.android.apps.youtube.music:id/text1" and @text="Library"]'
LIBRARY_BUTTON_XPATH2 = "//*[@resource-id='com.google.android.apps.youtube.music:id/pivot_bar']//*[@text='Library']"

# SEARCH BUTTON
SEARCH_BUTTON_XPATH1 = '//android.widget.ImageButton[@content-desc="Search"]'
SEARCH_BUTTON_XPATH2 = "//*[@resource-id='com.google.android.apps.youtube.music:id/action_search_button']"

# SEARCH INPUT
SEARCH_INPUT_XPATH1 = '//android.widget.EditText[@resource-id="com.google.android.apps.youtube.music:id/search_edit_text"]'
SEARCH_INPUT_XPATH2 = "//*[@resource-id='com.google.android.apps.youtube.music:id/search_edit_text']"

# SEARCH_DIRECTOR
YT_MUSIC_SEARCH_DIRECTORY_XPATH = '//android.widget.LinearLayout[@content-desc="YT Music"]'
LIBRARY_SEARCH_DIRECTORY_XPATH = '//android.widget.LinearLayout[@content-desc="Library"]'

# SEARCH FILTERS
SEARCH_FILTERS_BOX_XPATH1  = "//*[@resource-id='com.google.android.apps.youtube.music:id/chip_cloud']"
SEARCH_FILTERS_BOX_XPATH2 = '//android.support.v7.widget.RecyclerView[@resource-id="com.google.android.apps.youtube.music:id/chip_cloud"]'

SEARCH_FILTERS_ARTISTS_XPATH1 = "//*[@resource-id='com.google.android.apps.youtube.music:id/chip_cloud_chip_text' and @text='Artists']" 
SEARCH_FILTERS_ARTISTS_XPATH2 = '//android.support.v7.widget.RecyclerView[@resource-id="com.google.android.apps.youtube.music:id/chip_cloud"]//*[@text="Artists"]'

SEARCH_FILTERS_ARTISTS_XPATH1 = "//*[@resource-id='com.google.android.apps.youtube.music:id/chip_cloud_chip_text' and @text='Artists']" 
SEARCH_FILTERS_ARTISTS_XPATH2 = '//android.support.v7.widget.RecyclerView[@resource-id="com.google.android.apps.youtube.music:id/chip_cloud"]//*[@text="Artists"]'

SEARCH_FILTERS_PROFILES_XPATH1 = "//*[@resource-id='com.google.android.apps.youtube.music:id/chip_cloud_chip_text' and @text='Profiles']" 
SEARCH_FILTERS_PROFILES_XPATH2 = '//android.support.v7.widget.RecyclerView[@resource-id="com.google.android.apps.youtube.music:id/chip_cloud"]//*[@text="Profiles"]'

SEARCH_FILTERS_ALBUMS_XPATH1 = "//*[@resource-id='com.google.android.apps.youtube.music:id/chip_cloud_chip_text' and @text='Albums']" 
SEARCH_FILTERS_ALBUMS_XPATH2 = "//android.support.v7.widget.RecyclerView[@resource-id='com.google.android.apps.youtube.music:id/chip_cloud']//*[@text='Albums']"

SEARCH_FILTERS_SONGS_XPATH1 = "//*[@resource-id='com.google.android.apps.youtube.music:id/chip_cloud_chip_text' and @text='Songs']" 
SEARCH_FILTERS_SONGS_XPATH2 = "//android.support.v7.widget.RecyclerView[@resource-id='com.google.android.apps.youtube.music:id/chip_cloud']//*[@text='Songs']"

SEARCH_FILTERS_COMMUNITY_PL_XPATH1 = "//*[@resource-id='com.google.android.apps.youtube.music:id/chip_cloud_chip_text' and @text='Community playlists']" 
SEARCH_FILTERS_COMMUNITY_PL_XPATH2 = "//android.support.v7.widget.RecyclerView[@resource-id='com.google.android.apps.youtube.music:id/chip_cloud']//*[@text='Community playlists']"

SEARCH_FILTERS_FEATURED_PL_XPATH1 = "//*[@resource-id='com.google.android.apps.youtube.music:id/chip_cloud_chip_text' and @text='Featured playlists']" 
SEARCH_FILTERS_FEATURED_PL_XPATH2 = "//android.support.v7.widget.RecyclerView[@resource-id='com.google.android.apps.youtube.music:id/chip_cloud']//*[@text='Featured playlists']"

# SEARCH_GO_BACK
SEARCH_GO_BACK_BUTTON_XPATH = '//android.widget.ImageView[@content-desc="Search back"]'

# MUSIC GO BACK
MUSIC_GO_BACK_BUTTON_XPATH = '//android.widget.ImageButton[@content-desc="Back"]'


# SEARH RESULTS
SEARCH_RESULTS_BOX_XPATH = '//*[@resource-id="com.google.android.apps.youtube.music:id/results_list"]'
SELECT_TARGET_FROM_RESULTS = lambda matching_text: f"""//*[@resource-id='com.google.android.apps.youtube.music:id/results_list']
  //*[contains(
      translate(@text,'ABCDEFGHIJKLMNOPQRSTUVWXYZ','abcdefghijklmnopqrstuvwxyz'),
      '{matching_text.lower()}'
  )]
"""
## For Clicking 3 DOTS We Need To First Find The Localtion Of Target Div And TargetCoordinate
## rect gives x, y, width, height 
# rect = element.rect x, y, width, height = rect["x"], rect["y"], rect["width"], rect["height"] 
## endpoint coordinates (bottom-right corner) end_x = x + width end_y = y + 

SHUFFLE_PLAY_BUTTON_XPATH1 = '//android.view.ViewGroup[@content-desc="Shuffle play"]'
SHUFFLE_PLAY_BUTTON_XPATH2 = '//*[@resource-id="com.google.android.apps.youtube.music:id/results_list"]//*[contains(@content-desc, "Shuffle")]'
START_MIX_BUTTON_XPATH = '//android.view.ViewGroup[@content-desc="Start mix"]'
CLOSE_MUSIC_PROPERTIES_XPATH = '//android.view.ViewGroup[@content-desc="Close"]'

MUSIC_DOWN_BUTTON_XPATH = "//*[@resource-id='com.google.android.apps.youtube.music:id/toolbar_back_navigation']"

POPUP_RUNNING_MUSIC='//*[@resource-id="com.google.android.apps.youtube.music:id/mini_player"]'

In [ ]:
import random
import math



def generate_skipping_rates(
    count: int = 8,
    low_range: list = [42, 58],
    high_range: list = [100, 120],
    high_count: int = 1,
    avg_range: list = [55, 60]
):
    """
    Generate 'count' numbers with constraints:
    - Most numbers from low_range
    - 'high_count' numbers from high_range
    - Average must fall within avg_range
    """

    while True:
        # Step 1: Pick random indices for high numbers
        high_indices = random.sample(range(count), high_count)

        # Step 2: Generate numbers in low range
        nums = random.sample(range(low_range[0], low_range[1]), count)

        # Step 3: Replace selected positions with high range numbers
        for idx in high_indices:
            nums[idx] = random.randint(high_range[0], high_range[1])

        # Step 4: Check average constraint
        avg = sum(nums) / count
        if avg_range[0] <= avg <= avg_range[1]:
            return nums, avg
  




BASE_WIDTH = 1080
BASE_HEIGHT = 2280

def random_point_in_circle(center_x, center_y, radius, device_width, device_height):
    """
    Generate random point inside circle (scaled to device resolution).
    Shrinks radius by 1% to stay inside boundaries.
    """
    # shrink radius by 1%
    radius = radius * 0.99

    # random polar coordinates
    r = radius * math.sqrt(random.random())  # uniform distribution inside circle
    theta = random.uniform(0, 2 * math.pi)

    # point relative to center
    x = center_x + r * math.cos(theta)
    y = center_y + r * math.sin(theta)

    # scale to current resolution
    scaled_x = int(x / BASE_WIDTH * device_width)
    scaled_y = int(y / BASE_HEIGHT * device_height)

    return scaled_x, scaled_y


def random_point_in_rectangle(left, top, right, bottom, device_width, device_height):
    """
    Generate random point inside rectangle defined by (left, top) and (right, bottom).
    Scales coordinates from baseline (1080x2280) to current device resolution.
    """
    # shrink boundaries by 1% to stay inside
    width = right - left
    height = bottom - top
    shrink_x = width * 0.01
    shrink_y = height * 0.01

    # random point inside shrunken rectangle
    x = random.uniform(left + shrink_x, right - shrink_x)
    y = random.uniform(top + shrink_y, bottom - shrink_y)

    # scale to current resolution
    scaled_x = int(x / BASE_WIDTH * device_width)
    scaled_y = int(y / BASE_HEIGHT * device_height)

    return scaled_x, scaled_y



def get_shuffle_btn_coord(device_width, device_height):
    center_x, center_y = 75, 1760  # approx center from given coords
    radius = 60  # half of width/height (120/2)
    return random_point_in_circle(center_x, center_y, radius, device_width, device_height)


def get_back_btn_coord(device_width, device_height):
    center_x, center_y = 280, 1760
    radius = 70  # half of 140
    return random_point_in_circle(center_x, center_y, radius, device_width, device_height)

def get_play_btn_coord(device_width, device_height):
    center_x, center_y = 535, 1770
    radius = 90  # half of 180
    return random_point_in_circle(center_x, center_y, radius, device_width, device_height)

def get_next_btn_coord(device_width, device_height):
    center_x, center_y = 785, 1760
    radius = 70  # half of 140
    return random_point_in_circle(center_x, center_y, radius, device_width, device_height)

def get_loop_btn_coord(device_width, device_height):
    center_x, center_y = 990, 1760
    radius = 60  # half of 120
    return random_point_in_circle(center_x, center_y, radius, device_width, device_height)


def get_like1_coord(device_width, device_height):
    center_x, center_y = 130, 1465   # approx center from given coords
    radius = 40                      # half of width/height (80/2)
    return random_point_in_circle(center_x, center_y, radius, device_width, device_height)


def get_dislike_btn_coord(device_width, device_height):
    center_x, center_y = 290, 1465   # approx center from given coords
    radius = 40                      # half of width/height (80/2)
    return random_point_in_circle(center_x, center_y, radius, device_width, device_height)

def get_random_click_coord(device_width, device_height):
    # Rectangle defined by your coordinates
    left, top = 660, 1250
    right, bottom = 1000, 1350
    return random_point_in_rectangle(left, top, right, bottom, device_width, device_height)



if __name__ == "__main__":
    numbers, average = generate_skipping_rates()
    print("Numbers:", numbers)
    print("Average:", average)


In [16]:
import os
import random
import subprocess
# from app.config import Config


class SEARCH_BY_ARTIST:

    def __init__(self, driver):
        self.driver = driver
        self.current_path = os.getcwd()
        

    def start_search_by_artist_stream(self, artist_name): 
        try:
            
            screenWidth, screenHeight = get_screen_resolution(self.driver)

            # STEP 1: Click Home Tab...
            print("--> Clicking Home Tab...")
            try: WebDriverWait(self.driver, timeout=10).until(EC.presence_of_element_located((By.XPATH, HOME_BUTTON_XPATH1))).click()
            except: 
                try: WebDriverWait(self.driver, timeout=10).until(EC.presence_of_element_located((By.XPATH, HOME_BUTTON_XPATH2))).click()
                except Exception as e:  raise Exception(f"Failed to Click Home Tab... \n{e}")

            
            

            # STEP 2: Clicking Search Icon...
            print("--> Clicking Search Icon...") 
            try: WebDriverWait(self.driver, timeout=10).until(EC.presence_of_element_located((By.XPATH, SEARCH_BUTTON_XPATH1))).click()
            except: 
                try: WebDriverWait(self.driver, timeout=10).until(EC.presence_of_element_located((By.XPATH, SEARCH_BUTTON_XPATH2))).click()
                except Exception as e: raise Exception(f"Failed to Click Search Icon... \n{e}")
                
        
            
            
            # STEP 3: Typing Artist Name...
            print(f"--> Typing Artist Name [{artist_name.replace(' ', '_').upper()}]...")
            search_inp_obj = None
            try: 
                WebDriverWait(self.driver, timeout=10).until(EC.presence_of_element_located((By.XPATH, SEARCH_INPUT_XPATH1)))
                search_inp_obj = self.driver.find_element(By.XPATH, SEARCH_INPUT_XPATH1)
            except: 
                try: 
                    WebDriverWait(self.driver, timeout=5).until(EC.presence_of_element_located((By.XPATH, SEARCH_INPUT_XPATH2)))
                    search_inp_obj = self.driver.find_element(By.XPATH, SEARCH_INPUT_XPATH2)
                except Exception as e: raise Exception(f"Failed to Find Search Input Box... \n{e}")
            
            try: 
                if search_inp_obj: search_inp_obj.click()
                else: raise Exception("Search Input Box Not Found...")
                search_inp_obj.send_keys(artist_name)
                self.driver.execute_script("mobile: performEditorAction", {
                    "action": "search"
                })
            except Exception as e: raise Exception(f"Failed to Type Artist Name... \n{e}")

            


            # STEP 4: Clicking 'Shuffle' Button From Results...
            print("--> Clicking 'Shuffle' Button From Results...") 
            try: WebDriverWait(self.driver, timeout=10).until(EC.presence_of_element_located((By.XPATH, SHUFFLE_PLAY_BUTTON_XPATH1))).click()
            except: 
                try: WebDriverWait(self.driver, timeout=5).until(EC.presence_of_element_located((By.XPATH, SHUFFLE_PLAY_BUTTON_XPATH2))).click()
                except Exception as e: raise Exception(f"Failed to Click 'Shuffle' Button From Results... \n{e}")

            
            # STEP 5: Clicking 'Shuffle' + Reat Button...
            try:
                time.sleep(8)
                shuffle_btn_x, shuffle_btn_y = get_shuffle_btn_coord(screenWidth, screenHeight)
                print(f"--> Clicking 'Shuffle' button [{shuffle_btn_x}, {shuffle_btn_y}]...") 
                tapOnScreenCoord(self.driver, shuffle_btn_x, shuffle_btn_y)
                time.sleep(5)

                loop_btn_x, loop_btn_y = get_loop_btn_coord(screenWidth, screenHeight)
                print(f"--> Clicking 'Loop' button [{loop_btn_x}, {loop_btn_y}]...") 
                tapOnScreenCoord(self.driver, loop_btn_x, loop_btn_y)
                time.sleep(5)

            except Exception as e:
                print(e)
                raise Exception(str(e)) 


            print("--> Skipping First Song...") 
            next_btn_x, next_btn_y = get_next_btn_coord(screenWidth, screenHeight)
            tapOnScreenCoord(self.driver, next_btn_x, next_btn_y)
            time.sleep(1.5)


            # STEP 4: Generating Rates And Playing Musics...
            sets = 1
            
            while True:
                
                random_skipping_rates, rates_avg = generate_skipping_rates(
                    SKIPPING_RATES_COUNT,
                    SKIPPING_LOW_RANGE,
                    SKIPPING_HIGH_RANGE,
                    SKIPPING_HIGH_COUNT,
                    SKIPPING_AVG_RANGE
                )
                print(f"--> Generated Skipping Rates: [{random_skipping_rates}] Average: [{rates_avg}] Sets: [{sets}]...")
                
                for rate_index, skip_time in enumerate(random_skipping_rates):
                    start = time.monotonic()

                    next_btn_x, next_btn_y = get_next_btn_coord(screenWidth, screenHeight)
                    tapOnScreenCoord(self.driver, next_btn_x, next_btn_y)
                    time.sleep(1.5)

                    while time.monotonic() - start < skip_time:
                        time.sleep(0.5)  # check every half second
                    
                    print(f"--> --> [{rate_index+1}] Listened Song: [{skip_time}] Seconds")

                    
                sets += 1
                break

            
            
        except Exception as e: raise Exception(str(e))
        finally: pass

In [17]:
driver = create_driver(device_id, PACKAGE_ID, ACTIVITY)
SEARCH_BY_ARTIST_ = SEARCH_BY_ARTIST(driver)

In [ ]:
SEARCH_BY_ARTIST_.start_search_by_artist_stream("Fjalbaos")

In [ ]:
driver.quit()